# Similar stays: evaluation

`backend/app/modules/ml/similar.py` ranks stays in the same city by TF-IDF cosine similarity
over amenities, star class, rating band and price band. With no click data to learn from yet,
it is checked for the properties a traveller would expect: recommended stays should usually
share the star class and price band, and share distinctive amenities more than random pairs.

In [1]:
import json, sys, tempfile, datetime
from collections import defaultdict
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd
from app.core.config import get_settings
from app.datagen.generator import GeneratorOptions, generate
from app.modules.ingestion.mapping import load_mapping
from app.modules.ingestion.pipeline import prepare
from app.modules.ingestion.rules import load_rules, load_vocabulary
from app.modules.ingestion.sources import open_source
from app.modules.ml.similar import HotelProfile, similar_hotels
tmp = Path(tempfile.mkdtemp()) / "demo.zip"
generate(tmp, GeneratorOptions(seed=7, start=date(2026, 11, 13), days=3, dirty_rate=0))
s = get_settings()
p = prepare(open_source(tmp, max_members=10, max_uncompressed_mb=900), load_mapping(s.config_dir, "synthetic-v1"),
            load_rules(s.config_dir), load_vocabulary(s.config_dir), catalog_airports=lambda: [])
hotels, rooms = p.hotels, p.room_types
cheapest = rooms.groupby("hotel_id").nightly_rate_minor.min()
print(len(hotels), "hotels in", hotels.city_key.nunique(), "cities")

297 hotels in 25 cities


In [2]:
rng = np.random.default_rng(7)
same_star, same_band, jaccard_similar, jaccard_random = [], [], [], []
def jac(a, b): return len(a & b) / max(len(a | b), 1)
for city, group in hotels.groupby("city_key"):
    profiles = [HotelProfile(h.id, tuple(h.amenity_codes), h.star_rating, h.guest_rating, cheapest.get(h.id))
                for h in group.itertuples()]
    if len(profiles) < 4:
        continue
    prices = np.array([pr.nightly_price or 0 for pr in profiles]); edges = np.quantile(prices, [0.2, 0.4, 0.6, 0.8])
    band = {pr.id: int(np.searchsorted(edges, pr.nightly_price or 0)) for pr in profiles}
    by_id = {pr.id: pr for pr in profiles}
    for pr in profiles:
        for n in similar_hotels(pr.id, profiles, k=3):
            other = by_id[n.id]
            same_star.append(int(pr.star_rating) == int(other.star_rating))
            same_band.append(abs(band[pr.id] - band[n.id]) <= 1)
            jaccard_similar.append(jac(set(pr.amenities), set(other.amenities)))
            rand = profiles[rng.integers(len(profiles))]
            jaccard_random.append(jac(set(pr.amenities), set(rand.amenities)))
metrics = {"same_star_class": float(np.mean(same_star)), "within_one_price_band": float(np.mean(same_band)),
           "amenity_jaccard_similar": float(np.mean(jaccard_similar)), "amenity_jaccard_random": float(np.mean(jaccard_random))}
print(metrics)
assert metrics["amenity_jaccard_similar"] > metrics["amenity_jaccard_random"]

{'same_star_class': 0.6992143658810326, 'within_one_price_band': 0.8249158249158249, 'amenity_jaccard_similar': 0.6246985369039241, 'amenity_jaccard_random': 0.5576163330372085}


In [3]:
card = {"name": "similar_hotels", "version": datetime.date.today().strftime("%Y.%m.%d"),
        "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "framework": "scikit-learn",
        "framework_version": __import__("sklearn").__version__, "dataset": "TripMind synthetic catalog",
        "features": ["amenities", "star_class", "rating_band", "price_band"], "metrics": metrics,
        "notes": "Computed on demand from the live catalog; this card records the evaluation."}
Path("artifacts").mkdir(exist_ok=True); Path("artifacts/model_card.json").write_text(json.dumps(card, indent=2))

609